# Nettoyage d'un dataset « sale »

## Contexte

Ce notebook simule un cas réel de **data cleaning** à partir d'un CSV volontairement imparfait.

Le fichier contient plusieurs formats de dates, plusieurs écritures des valeurs manquantes, des nombres stockés comme texte, des doublons et des chaînes incohérentes.

L'objectif est de documenter chaque décision et de produire une version propre du fichier.


## 1. Import des bibliothèques

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)


## 2. Chargement du dataset

J'utilise `read_csv()` avec `;` comme séparateur. Les valeurs `N/A`, `NaN`, `null` et les cellules vides sont considérées comme des valeurs manquantes.

**Décision :** utiliser une représentation unique des valeurs manquantes facilite tous les traitements suivants.


In [ ]:
missing_values = ["", "NaN", "N/A", "null", "NULL", "None"]

df = pd.read_csv(
    "dataset_sale.csv",
    sep=";",
    na_values=missing_values,
    keep_default_na=True,
)

display(df)


## 3. Diagnostic initial

Avant de nettoyer, je regarde les dimensions, les types et les valeurs manquantes. Cela permet de mesurer les problèmes présents dans le fichier.


In [ ]:
print("Dimensions :", df.shape)
print("\nTypes :")
display(df.dtypes)
print("\nValeurs manquantes :")
display(df.isna().sum())


## 4. Suppression des doublons

Le doublon de Julien Faure est une copie exacte d'une ligne déjà présente.

**Décision :** supprimer uniquement les doublons exacts avec `drop_duplicates()`. Une ligne seulement similaire n'aurait pas été supprimée, car elle pourrait représenter une vraie observation différente.


In [ ]:
print("Doublons :", df.duplicated().sum())
df = df.drop_duplicates().copy()
print("Dimensions après nettoyage :", df.shape)


## 5. Nettoyage des colonnes texte

Je retire les espaces inutiles avec `str.strip()`.

Pour `statut`, `ACTIVE` et `active` représentent la même information. Je normalise donc les valeurs vers `actif`, `inactif` et `inconnu`.

**Décision :** une catégorie `inconnu` est préférable à une suppression lorsque l'information manque mais que le reste de la ligne reste exploitable.


In [ ]:
for col in ["nom", "ville", "statut", "email"]:
    df[col] = df[col].astype("string").str.strip()

df["statut"] = (
    df["statut"].str.lower()
    .replace({"active": "actif", "inactive": "inactif"})
    .fillna("inconnu")
)

df["ville"] = df["ville"].fillna("inconnue")

display(df[["nom", "ville", "statut", "email"]])


## 6. Normalisation des dates

Les dates utilisent plusieurs formats (`2026-01-15`, `15/02/2026`, `10.05.2026`, etc.).

**Décision :** convertir toutes les dates en objets datetime avec `pd.to_datetime()`. `errors="coerce"` transforme une date impossible à interpréter en valeur manquante au lieu de provoquer une erreur.

Pour l'export final, j'utilise `YYYY-MM-DD`, un format ISO non ambigu.


In [ ]:
df["date_achat"] = pd.to_datetime(
    df["date_achat"],
    errors="coerce",
    format="mixed",
    dayfirst=True,
)

display(df[["id", "date_achat"]])

df["date_achat"] = df["date_achat"].dt.strftime("%Y-%m-%d")


## 7. Conversion de l'âge

L'âge est une donnée numérique mais certaines valeurs sont du texte ou représentent une absence de donnée.

**Décision :** convertir avec `pd.to_numeric(errors="coerce")`.

Les âges manquants sont remplacés par la médiane. L'âge n'est pas indispensable à la transaction et la médiane évite de supprimer des lignes utiles tout en étant moins sensible aux valeurs extrêmes que la moyenne.


In [ ]:
df["age"] = pd.to_numeric(df["age"], errors="coerce")
age_median = df["age"].median()
df["age"] = df["age"].fillna(age_median)

print("Médiane utilisée :", age_median)
display(df[["nom", "age"]])


## 8. Conversion du montant

Le montant est particulièrement sale : virgule décimale, espace pour les milliers, point décimal ou valeurs manquantes.

**Décision :** supprimer les espaces, retirer la virgule lorsqu'elle sert de séparateur de milliers (`1,500.75`), remplacer la virgule décimale par un point, puis convertir avec `to_numeric()`.


In [ ]:
montant = df["montant"].astype("string").str.replace(" ", "", regex=False)
montant = montant.str.replace(
    r"(?<=\d),(?=\d{3}\.)",
    "",
    regex=True,
)
montant = montant.str.replace(",", ".", regex=False)

df["montant"] = pd.to_numeric(montant, errors="coerce")
display(df[["nom", "montant"]])


## 9. Suppression des lignes sans montant ou sans date

Le `montant` est indispensable pour une analyse financière : je ne peux pas inventer une valeur absente. Je supprime donc ces lignes.

La `date_achat` est indispensable pour une analyse temporelle. Si elle reste impossible à convertir, je supprime également la ligne.

**Décision :** supprimer uniquement ces lignes obligatoires plutôt que d'inventer des données.


In [ ]:
avant = len(df)
df = df.dropna(subset=["montant"]).copy()
print("Lignes supprimées sans montant :", avant - len(df))

avant = len(df)
df = df.dropna(subset=["date_achat"]).copy()
print("Lignes supprimées sans date :", avant - len(df))


## 10. Vérification de la version propre

Je contrôle les dimensions, les types et les valeurs manquantes avant l'export.


In [ ]:
print("Dimensions finales :", df.shape)
print("\nTypes finaux :")
display(df.dtypes)

print("\nValeurs manquantes :")
display(df.isna().sum())

display(df)


## 11. Export du dataset propre

J'enregistre le résultat dans `dataset_propre.csv`.

**Décision :** conserver le séparateur `;` et utiliser explicitement UTF-8 pour préserver les caractères accentués.


In [ ]:
df.to_csv(
    "dataset_propre.csv",
    sep=";",
    index=False,
    encoding="utf-8",
)

print("Fichier propre créé : dataset_propre.csv")


## 12. Contrôles de qualité

Avant de considérer le nettoyage terminé, je vérifie quelques règles simples : identifiants uniques, âge et montant numériques, dates présentes et montant présent.


In [ ]:
assert not df["id"].duplicated().any()
assert pd.api.types.is_numeric_dtype(df["age"])
assert pd.api.types.is_numeric_dtype(df["montant"])
assert df["date_achat"].notna().all()
assert df["montant"].notna().all()

print("Tous les contrôles de qualité sont OK.")


# Conclusion

Le dataset initial comportait plusieurs problèmes typiques d'un fichier réel.

### Nettoyages réalisés

- standardisation des valeurs manquantes ;
- suppression des doublons exacts ;
- nettoyage des chaînes de caractères ;
- normalisation des statuts ;
- conversion des dates vers `YYYY-MM-DD` ;
- conversion de l'âge en numérique et remplacement des âges manquants par la médiane ;
- conversion des montants en nombres ;
- suppression des lignes sans montant ou sans date ;
- export d'un fichier propre.

### Bonus

Le script `nettoyage_dataset.py` reprend ces règles dans une fonction réutilisable et ajoute des logs pour suivre les opérations.
